# 🥉 Bronze Layer - Production Data Ingestion
## AWS S3 External Storage via Unity Catalog

**Purpose**: Ingest Production schema tables from source to Bronze layer

**Source**: `adventureworks_live_conn_catalog.production.*`
**Target**: `workspace.`workspace-adventureworks-bronze`` (S3)
**Tables**: 25 tables (~350K rows)

**Key Tables**:
- Product
- ProductCategory
- ProductSubcategory
- ProductModel
- WorkOrder
- TransactionHistory
- BillOfMaterials
- Location

**Transformation**: Add audit columns (ingestion_timestamp, source_system)

In [0]:
from pyspark.sql.functions import current_timestamp, lit
from datetime import datetime
import time

# Configuration
SOURCE_CATALOG = "adventureworks_live_conn_catalog"
SOURCE_SCHEMA = "production"
TARGET_SCHEMA = "workspace.`workspace-adventureworks-bronze`"
SOURCE_SYSTEM = "AdventureWorks_OLTP"

print("=" * 80)
print("🥉 BRONZE LAYER - PRODUCTION DATA INGESTION")
print("=" * 80)
print(f"Start Time: {datetime.now()}")
print(f"Source: {SOURCE_CATALOG}.{SOURCE_SCHEMA}")
print(f"Target: {TARGET_SCHEMA} (AWS S3 External Storage)")
print(f"Source System: {SOURCE_SYSTEM}")
print("=" * 80)
print()

ingestion_results = []

In [0]:
def ingest_table_to_bronze(source_table_name, target_table_name=None):
    """
    Ingest a single table from source to Bronze with audit columns
    """
    if target_table_name is None:
        target_table_name = source_table_name
    
    source_full = f"{SOURCE_CATALOG}.{SOURCE_SCHEMA}.{source_table_name}"
    target_full = f"{TARGET_SCHEMA}.{target_table_name}"
    
    result = {
        "source_table": source_table_name,
        "target_table": target_table_name,
        "status": "failed",
        "row_count": 0,
        "duration_seconds": 0,
        "error": None
    }
    
    try:
        start_time = time.time()
        print(f"\n⏳ Ingesting: {source_table_name}...", end=" ")
        
        df = spark.table(source_full)
        
        df_bronze = (df
            .withColumn("ingestion_timestamp", current_timestamp())
            .withColumn("source_system", lit(SOURCE_SYSTEM))
        )
        
        (df_bronze
            .write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(target_full)
        )
        
        row_count = df_bronze.count()
        duration = time.time() - start_time
        
        result.update({
            "status": "success",
            "row_count": row_count,
            "duration_seconds": round(duration, 2)
        })
        
        print(f"✅ {row_count:,} rows in {duration:.2f}s")
        
    except Exception as e:
        duration = time.time() - start_time
        result.update({
            "error": str(e),
            "duration_seconds": round(duration, 2)
        })
        print(f"❌ Failed: {str(e)}")
    
    return result

In [0]:
# Get list of tables and ingest
print("🔍 Discovering Production tables...")

try:
    production_tables = spark.sql(f"SHOW TABLES IN {SOURCE_CATALOG}.{SOURCE_SCHEMA}").collect()
    table_names = [row.tableName for row in production_tables]
    print(f"✅ Found {len(table_names)} tables in Production schema")
    print("\nTables to ingest:")
    for i, table in enumerate(table_names, 1):
        print(f"  {i}. {table}")
except Exception as e:
    print(f"❌ Error discovering tables: {str(e)}")
    raise

# Ingest all tables
print("\n" + "=" * 80)
print("🚀 STARTING INGESTION")
print("=" * 80)

overall_start = time.time()

for table_name in table_names:
    result = ingest_table_to_bronze(table_name)
    ingestion_results.append(result)

overall_duration = time.time() - overall_start

print("\n" + "=" * 80)
print("🏁 INGESTION COMPLETE")
print("=" * 80)
print(f"Total Duration: {overall_duration:.2f}s ({overall_duration/60:.2f} minutes)")

In [0]:
from pyspark.sql import Row

print("\n📊 INGESTION SUMMARY\n")

success_count = sum(1 for r in ingestion_results if r["status"] == "success")
failed_count = sum(1 for r in ingestion_results if r["status"] == "failed")
total_rows = sum(r["row_count"] for r in ingestion_results)

print(f"Tables Processed: {len(ingestion_results)}")
print(f"  ✅ Success: {success_count}")
print(f"  ❌ Failed: {failed_count}")
print(f"Total Rows Ingested: {total_rows:,}")

summary_data = [
    Row(
        source_table=r["source_table"],
        target_table=r["target_table"],
        status=r["status"],
        row_count=r["row_count"],
        duration_seconds=r["duration_seconds"],
        error=r["error"]
    )
    for r in ingestion_results
]

summary_df = spark.createDataFrame(summary_data)
print("\nDetailed Results:")
display(summary_df.orderBy("status", "source_table"))

if failed_count > 0:
    print("\n⚠️ FAILED TABLES:")
    for r in ingestion_results:
        if r["status"] == "failed":
            print(f"  - {r['source_table']}: {r['error']}")

print("\n" + "=" * 80)
if failed_count == 0:
    print("✅ BRONZE PRODUCTION INGESTION COMPLETED SUCCESSFULLY")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit("SUCCESS")
else:
    print("⚠️ BRONZE PRODUCTION INGESTION COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"Completion Time: {datetime.now()}")
    dbutils.notebook.exit(f"PARTIAL_SUCCESS: {failed_count} tables failed")